# WA Health establishments: a 49-site feature table

## What is this data source?

This notebook uses the Western Australian Department of Health **Health Hospitals (HEALTH-001)** dataset published through Data WA. The official source contains the locations of public and private **Acute Hospitals** and **Nursing Posts** in Western Australia and is updated annually.

## Which API is used?

This notebook uses the public **ArcGIS REST Feature Layer API** provided through the Western Australian Land Information Platform (SLIP). The layer endpoint is:

```text
https://public-services.slip.wa.gov.au/public/rest/services/
SLIP_Public_Services/Health_WFS/MapServer/1
```

The notebook first requests the layer metadata with `?f=pjson` and confirms that the endpoint type is `Feature Layer`. It then calls the layer's `/query` endpoint once with `where=1=1`, `outFields=*`, `returnGeometry=true`, `outSR=4326`, and `f=geojson` to download all statewide records and point geometry. The API is public and the notebook does not use an API key. A dated local GeoJSON snapshot is reused after the first download so repeated model runs do not silently change when the live dataset is updated.

The source gives us an establishment ID and name, point coordinates, whether the establishment is open, its official category, its Public/Private type, and a bed-count field. It does **not** cover every GP, clinic, pharmacy, emergency service, opening hour, service quality, travel route, or a person's ability to obtain care. For that reason, the extracted variables are named `health_establishment` features rather than broad “healthcare accessibility” features.

## What does this notebook produce?

The notebook reads `site_id`, latitude, and longitude from the authoritative `Model_Data` sheet in `model_dataset.xlsx`. It finds the nearest open establishment for every model site and writes a separate one-row-per-`site_id` feature workbook. Targets and unrelated model columns are not copied into that workbook.

| Final output field | Plain meaning | Intended use |
|---|---|---|
| `health_nearest_establishment_distance_m` | Straight-line geodesic distance, in metres, from the model site to the nearest open Acute Hospital or Nursing Post | The only first-round candidate model feature |
| `health_nearest_establishment_category` | Official category of the nearest establishment: `Acute Hospital` or `Nursing Post` | Interpretation and diagnostic checks; not automatically included in the model |
| `health_nearest_establishment_type` | Official type of the nearest establishment: `Public` or `Private` | Interpretation and diagnostic checks; it does not guarantee access |
| `health_nearest_establishment_id` | Official identifier of the nearest establishment | Audit and reproducibility only; not a model feature |
| `health_nearest_establishment_name` | Official name of the nearest establishment | Human review and spot checks only; not a model feature |
| `health_data_status` | Whether extraction was `matched`, `no_open_record`, `invalid_coordinate`, or `source_failed` | Data-quality control only; not a behavioural model feature |

The bed-count field is not extracted because its zeros and values have not been established as comparable measures of current capacity across facility categories. Radius counts are also excluded because the statewide source is sparse and an arbitrary radius would create many zeros. The output is designed for a later `validate="one_to_one"` merge with other teams' feature tables.


## 1. Configuration and reproducibility

The dated GeoJSON snapshot is an immutable external input. If it already exists, the notebook runs without contacting the service. If it is absent, the code first confirms that the REST endpoint is a `Feature Layer`, then downloads all statewide records once. Change `SNAPSHOT_DATE` only when intentionally creating and reviewing a new source version.


In [22]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill


def find_project_root() -> Path:
    # Find the project repository root on the host or in the container.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Project root not found from {current}. Expected requirements.txt and data/."
    )


PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from pedflow.context.config import load_context_config
from pedflow.context.sources.wa_health_establishments import (
    DISTANCE_REGRESSION_ATOL_M,
    EXACT_REGRESSION_COLUMNS,
    FEATURE_COLUMNS,
    acquire_or_load_source,
    build_features,
    make_feature_result,
    prepare_source,
    validate_source_specific,
)
from pedflow.context.validation import validate_and_normalise_sites


MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "02_feature_extraction"
FEATURE_PATH = OUTPUT_DIR / "02_04_haoqi_wa_health_features.xlsx"
PIPELINE_CONFIG_PATH = PROJECT_ROOT / "configs" / "context_features_v1.json"

pipeline_config = load_context_config(PIPELINE_CONFIG_PATH)
HEALTH_CONFIG = next(
    source_config
    for source_config in pipeline_config.sources
    if source_config.name == "wa_health_establishments"
)

if not MODEL_DATASET_PATH.is_file():
    raise FileNotFoundError(f"Required model input not found: {MODEL_DATASET_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root  :", PROJECT_ROOT)
print("Model input   :", MODEL_DATASET_PATH.relative_to(PROJECT_ROOT))
print("Source snapshot:", HEALTH_CONFIG.options["snapshot_path"])
print("Feature output:", FEATURE_PATH.relative_to(PROJECT_ROOT))
print("Shared source enabled:", HEALTH_CONFIG.enabled)


Project root  : /workspace
Model input   : data/processed/model_dataset.xlsx
Source snapshot: data/external_raw/wa_health_establishments/2026-08-24/health_hospitals_health_001.geojson
Feature output: 06_outputs/02_feature_extraction/02_04_haoqi_wa_health_features.xlsx
Shared source enabled: True


## 2. Read and validate the current model sites

`model_dataset.xlsx` is the project's authoritative modelling foundation. Feature extraction uses only `site_id`, WGS84 coordinates, and CRS. It does not filter on the targets. The present workbook has 49 unique sites and resolves the earlier missing Northampton record; therefore the older 48-row CSV precondition in `04_wa_health.md` is no longer applicable.


In [14]:
model_data = pd.read_excel(
    MODEL_DATASET_PATH,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)

site_columns = ["site_id", "latitude", "longitude", "coordinate_crs"]
sites = validate_and_normalise_sites(model_data[site_columns])

print(f"Model sites: {len(sites):,}")
print(f"Unique site_id values: {sites['site_id'].nunique():,}")
print(f"Valid EPSG:4326 coordinates: {len(sites):,}")
if "qa_status" in model_data.columns:
    print("qa_status distribution:")
    print(model_data["qa_status"].value_counts(dropna=False).to_string())
else:
    print("qa_status: not present in the current Model_Data schema")
print(sites.head().to_string(index=False))


Model sites: 49
Unique site_id values: 49
Valid EPSG:4326 coordinates: 49
qa_status: not present in the current Model_Data schema
  site_id   latitude  longitude coordinate_crs
250303-01 -32.866282 115.705163      EPSG:4326
250303-02 -33.359642 115.678041      EPSG:4326
250303-03 -33.360624 116.154738      EPSG:4326
250303-04 -34.238746 116.149491      EPSG:4326
250303-05 -32.628918 115.874612      EPSG:4326


## 3. Official dataset scope and metadata

The official [Data WA catalogue record](https://catalogue.data.wa.gov.au/dataset/health-establishments) identifies **Health Hospitals (HEALTH-001)** as a Department of Health dataset covering public and private acute hospitals and nursing posts in Western Australia. It says the list is updated annually. The catalogue currently reports data last updated on 8 July 2025, metadata last updated on 18 August 2026, source projection GDA2020, and “Licence not supplied”; the copyright notice is © Government of Western Australia 2025, Department of Health. The source therefore must not be relabelled as CC BY.

The verified [official Feature Layer](https://public-services.slip.wa.gov.au/public/rest/services/SLIP_Public_Services/Health_WFS/MapServer/1) exposes the fields used here: `estab_id` (Establishment ID), `establishm` (Establishment Name), `is_open` (Open), `category` (Category), `est` (Establishment Type), `no_of_beds` (Number Of Beds), `longitude`, and `latitude`. This source does **not** represent all GPs, clinics, pharmacies, emergency capability, opening hours, service capacity, travel time, or individual access.


## 4. Load one dated statewide snapshot

The live-service check is performed only when a snapshot must be created. The query requests all fields and point geometry in EPSG:4326 in a single statewide call. A cached snapshot is preferred for repeatability and to avoid silently changing feature values between model runs.



In [23]:
acquired_source = acquire_or_load_source(
    HEALTH_CONFIG,
    PROJECT_ROOT,
)

print("Source mode:", acquired_source.retrieval_method)
print("Snapshot date:", acquired_source.snapshot_date)
print("Source records:", len(acquired_source.payload.get("features", [])))
print("Snapshot SHA-256:", acquired_source.sha256)


Source mode: cached dated GeoJSON snapshot
Snapshot date: 2026-08-24
Source records: 153
Snapshot SHA-256: 9845003cef03997800a0eb5b0fcd38fc94a00f4ca5b8722d741fc21b9c30f99b


## 5. Source quality checks and accepted value domains

The extraction stops for an unexpected schema or value domain because silently accepting a changed source would alter the meaning of the feature. Counts are reported rather than hard-coded: an annual update may legitimately change the number of establishments. The current snapshot should be reviewed for duplicate IDs, missing point geometry, WA coordinate bounds, open status, category, and Public/Private type.

In [18]:
prepared_source = prepare_source(acquired_source, HEALTH_CONFIG)
health_source = prepared_source.records
open_health = prepared_source.eligible_records

print(f"Source records: {prepared_source.source_record_count:,}")
print(f"Open eligible records: {prepared_source.eligible_record_count:,}")
print(f"Duplicate establishment IDs: {health_source['estab_id'].duplicated().sum():,}")
print("Invalid point geometries: 0")
print(
    "Maximum attribute/geometry coordinate difference: "
    f"{prepared_source.maximum_coordinate_difference_deg:.6f}°"
)
for column in ["is_open", "category", "est"]:
    print(f"\n{column} distribution:")
    print(health_source[column].value_counts(dropna=False).to_string())
print("\nNumber Of Beds checks (excluded from features):")
print(
    {
        "missing": int(health_source["no_of_beds"].isna().sum()),
        "zero": int(health_source["no_of_beds"].eq(0).sum()),
        "median": float(health_source["no_of_beds"].median()),
        "maximum": float(health_source["no_of_beds"].max()),
    }
)


Source records: 153
Open eligible records: 153
Duplicate establishment IDs: 0
Invalid point geometries: 0
Maximum attribute/geometry coordinate difference: 0.000014°

is_open distribution:
is_open
Y    153

category distribution:
category
Acute Hospital    100
Nursing Post       53

est distribution:
est
Public     121
Private     32

Number Of Beds checks (excluded from features):
{'missing': 0, 'zero': 55, 'median': 11.0, 'maximum': 783.0}


## 6. Selected fields and modelling roles

| Output field | Meaning | Role in this 49-site project |
|---|---|---|
| `health_nearest_establishment_distance_m` | WGS84 ellipsoidal straight-line distance to the nearest `is_open == "Y"` record | The only first-baseline model candidate; compare models with and without it inside grouped/spatial CV |
| `health_nearest_establishment_category` | Official `Acute Hospital` or `Nursing Post` category | Diagnostic descriptor; do not automatically one-hot encode in this small sample |
| `health_nearest_establishment_type` | Official `Public` or `Private` type | Diagnostic descriptor; it does not imply that a person can access the service |
| `health_nearest_establishment_id` and `_name` | Official identifier and name | Audit and spot-check fields; never numeric or high-cardinality model inputs |
| `health_data_status` | `matched`, `no_open_record`, `invalid_coordinate`, or `source_failed` | Retrieval/QA status; not a behavioural predictor |

`no_of_beds` is rejected because zeros and facility categories are not yet comparable measures of current capacity. Radius counts are rejected because 153 statewide points are sparse and a chosen radius would be arbitrary. Network travel time, emergency capability, opening hours, and service quality are unavailable from this source and must not be inferred.


## 7. Calculate the nearest open establishment

The calculation uses the WGS84 ellipsoid through `pyproj.Geod`, not a degree-based Euclidean formula. This avoids UTM-zone management across statewide sites and is more accurate than a spherical approximation. It is still straight-line distance—not walking distance, driving distance, or travel time. Distance is rounded to 0.1 metre only for storage consistency; source-coordinate accuracy does not justify interpreting tiny differences.


In [24]:
features = build_features(sites, prepared_source, HEALTH_CONFIG)
validate_source_specific(features, sites, prepared_source, HEALTH_CONFIG)
health_result = make_feature_result(
    sites,
    features,
    acquired_source,
    prepared_source,
    HEALTH_CONFIG,
)

# Compare with the reviewed workbook before replacing it.
if FEATURE_PATH.is_file():
    reviewed_features = pd.read_excel(
        FEATURE_PATH,
        sheet_name="WA_Health_Features",
        dtype={
            "site_id": "string",
            "health_nearest_establishment_id": "string",
        },
    )
    pd.testing.assert_frame_equal(
        features.loc[:, EXACT_REGRESSION_COLUMNS].reset_index(drop=True),
        reviewed_features.loc[:, EXACT_REGRESSION_COLUMNS].reset_index(drop=True),
        check_dtype=False,
        check_exact=True,
    )
    np.testing.assert_allclose(
        features["health_nearest_establishment_distance_m"].to_numpy(float),
        reviewed_features["health_nearest_establishment_distance_m"].to_numpy(float),
        rtol=0,
        atol=DISTANCE_REGRESSION_ATOL_M,
    )
    print(
        "Regression check: exact identifiers/categories/statuses and distances "
        f"within {DISTANCE_REGRESSION_ATOL_M:.1f} m."
    )

print("Source version:", health_result.source_version)
print(features["health_data_status"].value_counts(dropna=False).to_string())
print(features.head().to_string(index=False))


Regression check: exact identifiers/categories/statuses and distances within 0.1 m.
Source version: Health-001 snapshot 2026-08-24 (9845003cef03)
health_data_status
matched    49
  site_id  health_nearest_establishment_distance_m health_nearest_establishment_category health_nearest_establishment_type health_nearest_establishment_id health_nearest_establishment_name health_data_status
250303-01                                  29476.9                        Acute Hospital                            Public                             412                   Harvey Hospital            matched
250303-02                                   2886.5                        Acute Hospital                            Public                             208                  Bunbury Hospital            matched
250303-03                                   1237.4                        Acute Hospital                            Public                             211                   Collie Hospital         

## 8. Save the independent feature workbook

The workbook contains only `site_id` and source-prefixed health fields. Header styling, filters, frozen headers, and explicit distance formatting improve review without adding a second metadata sheet or copying `Model_Data`. Source definitions and copyright cautions remain visible in this notebook.


In [25]:
features.to_excel(
    FEATURE_PATH,
    sheet_name="WA_Health_Features",
    index=False,
)

workbook = load_workbook(FEATURE_PATH)
worksheet = workbook["WA_Health_Features"]
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions
worksheet.sheet_view.showGridLines = True

header_fill = PatternFill("solid", fgColor="1F4E78")
for cell in worksheet[1]:
    cell.fill = header_fill
    cell.font = Font(color="FFFFFF", bold=True)
    cell.alignment = Alignment(vertical="center", wrap_text=True)

widths = {
    "A": 15,
    "B": 45,
    "C": 43,
    "D": 39,
    "E": 38,
    "F": 43,
    "G": 23,
}
for column, width in widths.items():
    worksheet.column_dimensions[column].width = width
worksheet.row_dimensions[1].height = 42

for cell in worksheet["B"][1:]:
    cell.number_format = "0.0"

workbook.properties.title = "WA Health establishment features for model sites"
workbook.properties.subject = "Independent site_id feature table for CITS5553"
workbook.properties.creator = "CITS5553 Group 5"
workbook.save(FEATURE_PATH)

print("Saved:", FEATURE_PATH.relative_to(PROJECT_ROOT))

Saved: 06_outputs/02_feature_extraction/02_04_haoqi_wa_health_features.xlsx


## 9. Verify coverage, inspect contrasting sites, and test the team merge contract

The checks below report the actual feature distribution, review selected peri-urban/regional/remote contexts, and prove that the output merges one-to-one without changing the 49-row model foundation. The current sample contains no central Perth observations, so it cannot support a genuine metropolitan generalisation check.



In [26]:
matched = features.loc[features["health_data_status"].eq("matched")].copy()

print("Matched distance summary (metres):")
print(
    matched["health_nearest_establishment_distance_m"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(1)
    .to_string()
)
print("\nNearest category distribution:")
print(
    matched["health_nearest_establishment_category"]
    .value_counts(dropna=False)
    .to_string()
)
print("\nNearest type distribution:")
print(
    matched["health_nearest_establishment_type"]
    .value_counts(dropna=False)
    .to_string()
)

check_site_ids = ["250303-01", "250303-03", "250341-04", "260335-28"]
manual_check = (
    model_data[["site_id", "survey_location_name", "latitude", "longitude"]]
    .merge(features, on="site_id", how="left", validate="one_to_one")
    .loc[lambda frame: frame["site_id"].isin(check_site_ids)]
)
print("\nSelected manual checks:")
print(manual_check.to_string(index=False))

# Replace any earlier health columns only for this explanatory merge check.
existing_health_columns = [
    column for column in FEATURE_COLUMNS[1:] if column in model_data.columns
]
modeling_df = model_data.drop(columns=existing_health_columns).merge(
    features,
    on="site_id",
    how="left",
    validate="one_to_one",
)

if len(modeling_df) != len(model_data):
    raise ValueError("The feature merge changed the Model_Data row count.")
if modeling_df["health_data_status"].isna().any():
    raise ValueError("At least one model site did not receive a health data status.")

print(f"\nRows after one-to-one merge: {len(modeling_df):,}")
if "qa_status" in modeling_df.columns:
    training_df = modeling_df.loc[modeling_df["qa_status"].eq("ready")].copy()
    print(f"Training rows with qa_status == 'ready': {len(training_df):,}")
else:
    print("Training subset not created: qa_status is absent from current Model_Data.")
print("First-baseline model candidate: health_nearest_establishment_distance_m")


Matched distance summary (metres):
count       49.0
mean      4601.2
std       7323.3
min         94.5
25%        671.3
50%       1136.1
75%       3199.3
max      29476.9

Nearest category distribution:
health_nearest_establishment_category
Acute Hospital    44
Nursing Post       5

Nearest type distribution:
health_nearest_establishment_type
Public     45
Private     4

Selected manual checks:
  site_id           survey_location_name   latitude  longitude  health_nearest_establishment_distance_m health_nearest_establishment_category health_nearest_establishment_type health_nearest_establishment_id health_nearest_establishment_name health_data_status
250303-01 Melville Mandurah Hwy Mandurah -32.866282 115.705163                                  29476.9                        Acute Hospital                            Public                             412                   Harvey Hospital            matched
250303-03          Coalfields Hwy Collie -33.360624 116.154738                  